In [ ]:
!pip install -q timm torch torchvision scikit-learn matplotlib opencv-python

: 

In [ ]:
pip install --upgrade pip

: 

In [ ]:
import torch
import timm
import sklearn

print("Torch version:", torch.__version__)
print("Timm version:", timm.__version__)
print("Sklearn version:", sklearn.__version__)
print("CUDA available (GPU):", torch.cuda.is_available())


: 

In [ ]:
import os

BASE_DIR = r"../My Final Year"
os.chdir(BASE_DIR)

print("Reset working directory to:")
print(os.getcwd())


: 

In [ ]:
PROJECT_ROOT = os.path.join(os.getcwd(), "rice-vit-fusion")

if not os.path.exists(PROJECT_ROOT):
    os.makedirs(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

print("Project root set to:")
print(os.getcwd())

: 

In [ ]:

folders = [
    "data",
    "data/train",
    "data/val",
    "data/test",
    "models",
    "results"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folder structure created:")
for root, dirs, files in os.walk(".", topdown=True):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

: 

In [ ]:
import zipfile
import os

zip_path = "archive.zip"
extract_path = "unzipped_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset unzipped successfully.")


Dataset unzipped successfully.


In [ ]:
import os
import shutil
import random

SOURCE_DIR = os.path.join("unzipped_dataset", "Rice_Image_Dataset")
TARGET_DIR = "data"

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(42)  

classes = os.listdir(SOURCE_DIR)

for cls in classes:
    cls_path = os.path.join(SOURCE_DIR, cls)
    if not os.path.isdir(cls_path):
        continue

    images = os.listdir(cls_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(TRAIN_RATIO * total)
    val_end = train_end + int(VAL_RATIO * total)

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for split, split_imgs in zip(
        ["train", "val", "test"],
        [train_imgs, val_imgs, test_imgs]
    ):
        split_dir = os.path.join(TARGET_DIR, split, cls)
        os.makedirs(split_dir, exist_ok=True)

        for img in split_imgs:
            src = os.path.join(cls_path, img)
            dst = os.path.join(split_dir, img)
            shutil.copy(src, dst)

    print(f"{cls}: {len(train_imgs)} train, {len(val_imgs)} val, {len(test_imgs)} test")

print("Dataset split completed successfully.")


: 

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms defined successfully.")


: 

In [ ]:
from torchvision.datasets import ImageFolder

train_dir = "data/train"
val_dir = "data/val"
test_dir = "data/test"

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
val_dataset = ImageFolder(root=val_dir, transform=val_test_transform)
test_dataset = ImageFolder(root=test_dir, transform=val_test_transform)

print("Datasets loaded successfully.")

print("Number of training images:", len(train_dataset))
print("Number of validation images:", len(val_dataset))
print("Number of test images:", len(test_dataset))

print("\nClass names:")
print(train_dataset.classes)

: 

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("DataLoaders created successfully.")

: 

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

: 

In [ ]:
import shutil
import os

cache_dir = os.path.expanduser("~/.cache/huggingface")

if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Hugging Face cache cleared.")
else:
    print("No Hugging Face cache found.")


: 

In [ ]:
NUM_CLASSES = 5

efficientnet_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

in_features = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

efficientnet_model = efficientnet_model.to(device)

print("EfficientNet-B0 (torchvision) created successfully.")


: 

In [ ]:
criterion = nn.CrossEntropyLoss()
print("Loss function defined:", criterion)

: 

In [ ]:
optimizer = torch.optim.Adam(
    efficientnet_model.parameters(),
    lr=0.0001
)

print("Optimizer defined:", optimizer)


: 

In [ ]:
!pip install -q tqdm

In [ ]:
from tqdm import tqdm

: 

In [ ]:
efficientnet_model.train()
running_loss = 0.0
correct = 0
total = 0
train_bar = tqdm(train_loader, desc="Training", leave=True)
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    outputs = efficientnet_model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    train_bar.set_postfix({
        "loss": f"{running_loss / (train_bar.n + 1):.4f}",
        "acc (%)": f"{100 * correct / total:.2f}"
    })
epoch_loss = running_loss / len(train_loader)
epoch_acc = 100 * correct / total
print(f"One-epoch training completed")
print(f"Training Loss: {epoch_loss:.4f}")
print(f"Training Accuracy: {epoch_acc:.2f}%")

: 